# Expression plots

Compare the same marker genes across cell groups. These examples use Scanpy’s packaged PBMC data.

Unreleased preview for Scanpy 1.11.5. Install from the project folder with `pip install -e '.[scanpy]'` and use that environment as your Jupyter kernel.


In [ ]:
import scanpy as sc
import genome_spy.integrations.scanpy as gsc

pbmc = sc.datasets.pbmc68k_reduced()
marker_genes = {
    "B-cell": ["CD79A", "MS4A1"],
    "Dendritic": ["FCER1A", "CST3"],
    "Monocytes": ["FCGR3A"],
    "NK": ["GNLY", "NKG7"],
    "Other": ["IGLL1"],
    "Plasma": ["IGJ"],
    "T-cell": ["CD3D"],
}

## Order the cell groups

Compute the group tree once with Scanpy. The plots below reuse it.


In [ ]:
sc.tl.dendrogram(pbmc, groupby="bulk_labels")

## Dotplot

Dot color shows mean expression, and dot size shows the percentage of cells expressing the gene. Hover for values; the crosshair marks its row and column.


In [ ]:
dots = gsc.pl.dotplot(
    pbmc,
    marker_genes,
    groupby="bulk_labels",
    dendrogram=True,
    zoom=True,
    highlight=True,
    width=240,
    height=320,
    show=False,
)
dots

## Matrixplot

Tile color shows mean expression, scaled from 0 to 1 for each gene. Scroll to zoom rows and drag to pan; gene columns stay fixed.


In [ ]:
means = gsc.pl.matrixplot(
    pbmc,
    marker_genes,
    groupby="bulk_labels",
    dendrogram=True,
    standard_scale="var",
    cmap="Blues",
    colorbar_title="Scaled mean",
    zoom=True,
    width=240,
    height=320,
    show=False,
)
means

## Heatmap

Each row is one cell; hover for its ID and expression value. Scroll to zoom into groups: `group_gap=1` separates them, and `show_values=True` reveals numbers as you zoom closer.


In [ ]:
cells = gsc.pl.heatmap(
    pbmc,
    marker_genes,
    groupby="bulk_labels",
    dendrogram=True,
    standard_scale="var",
    cmap="viridis",
    zoom=True,
    show_values=True,
    group_gap=1,
    width=300,
    height=400,
    show=False,
)
cells

Use `zoom="xy"` to zoom both axes. Gaps and numbers are optional: use `group_gap=0` and `show_values=False` to turn them off.



## Plot supplied numbers

Plot the group correlations that Scanpy computed above. Enable numbers explicitly with `show_values=True`; scroll to zoom rows.


In [ ]:
group_labels = pbmc.obs["bulk_labels"].cat.categories.tolist()
matrix = gsc.pl.matrix(
    pbmc.uns["dendrogram_bulk_labels"]["correlation_matrix"],
    xticks=group_labels,
    yticks=group_labels,
    color_map="viridis",
    show_values=True,
    zoom=True,
    width=300,
    height=300,
    show=False,
)
matrix

## Show group correlations

Reuse the stored correlations and group tree. Hover for a correlation; numbers are shown when they fit.


In [ ]:
correlations = gsc.pl.correlation_matrix(
    pbmc,
    "bulk_labels",
    show_values=True,
    width=300,
    height=300,
    show=False,
)
correlations

## Show the group tree

Display the same stored dendrogram on its own. Scanpy computed this tree in the first preparation cell.


In [ ]:
tree = gsc.pl.dendrogram(pbmc, "bulk_labels", width=420, height=240, show=False)
tree

## Cluster cells and genes

Clustermap clusters the selected cell expression values in Python. The colored strip shows cell types; hover or zoom to inspect the matrix.


In [ ]:
marker_names = [gene for genes in marker_genes.values() for gene in genes]
cluster_data = pbmc.raw[:, marker_names].to_adata()
clusters = gsc.pl.clustermap(
    cluster_data,
    obs_keys="bulk_labels",
    use_raw=False,
    cmap="viridis",
    yticklabels=False,
    zoom=True,
    width=300,
    height=400,
    show=False,
)
clusters

## Add cell counts to a dotplot

Use the builder to sort groups by cell count and add a count bar. The expression summary is still the same.


In [ ]:
dot_builder = (
    gsc.pl.DotPlot(
        pbmc,
        marker_genes,
        "bulk_labels",
        width=240,
        height=320,
        zoom=True,
        highlight=True,
    )
    .add_totals(sort="descending", color="steelblue")
    .style(cmap="viridis", largest_dot=140)
    .legend(colorbar_title="Mean expression", size_title="Cells (%)")
    .to_chart()
)
dot_builder

## Style a matrixplot

Chain settings with the matrixplot builder. Here, the stored tree orders the groups and the palette shows scaled means.


In [ ]:
matrix_builder = (
    gsc.pl.MatrixPlot(
        pbmc,
        marker_genes,
        "bulk_labels",
        standard_scale="var",
        width=240,
        height=320,
        zoom=True,
    )
    .add_dendrogram()
    .style(cmap="viridis")
    .legend(title="Scaled mean")
    .to_chart()
)
matrix_builder

:::{admonition} Data use and provenance
:class: note

The markers follow Scanpy’s [core plotting tutorial](https://scanpy.scverse.org/en/stable/tutorials/plotting/core.html). The packaged data contains 700 preprocessed PBMCs from [10x Genomics](https://www.10xgenomics.com/datasets/fresh-68-k-pbm-cs-donor-a-1-standard-1-1-0), licensed [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Credit 10x Genomics and Scanpy’s preprocessing contributors. Scanpy computes the group tree from the supplied PCA coordinates; GenomeSpy displays the selected expression values and group summaries.
:::
